# Field cancelation

This programm uses source code from the file ```MSR_map_coil.ipynb``` created by *Chiara Weckmann*.
___
Created on 05. Jun. 2026 by Gregor Bock 

(0378 1735; ge27doc)

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from mpl_toolkits.axes_grid1 import make_axes_locatable
import bfieldtools
from bfieldtools.mesh_calculus import laplacian_matrix
from bfieldtools.mesh_impedance import resistance_matrix, self_inductance_matrix
import scipy.constants
import scipy.sparse as sp
import scipy.sparse.linalg as spl
import os
mu_0=scipy.constants.mu_0
pi=scipy.constants.pi

# Self made functions and classes
import Ausgelagerte_Funktionen_Versuchsauswertung as fkt    # longer plots, file readouts and other larger functions are stored in this file to save space
from Externe_Classes import Coil_Layup, Mu_material         # Properties and methods of the Coil Layup and mu_metal are stored here (e.g. meshes, B-field by Biot-Savart, Stream_function calculation)

## Load Experiment Data

The data is loaded from a *map* directory, which has a *points* directory in which there are ```.npz``` (numpy-zip) files which hold the data for every point. The shape of the map is dertermined by the values in the ```.npz```-files as well as the total length, step_size and shift given in this block.

In [ ]:
# Specify folder path for .npz and points folder (.npz file should contain L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)
folder_path_points_no_current = r"D:\Studium\Physik\Bachelorarbeit\MSR-field_cancelation\Experiments\background_01_2026-05-13_10-48-57\map\points"

# If .npz file does not hold geometric track data, specify it here
L_x = 0.800                                         # length of the mapped volume in x-direction
L_y = 0.800                                         # length of the mapped volume in y-direction    
L_z = 0.400                                         # length of the mapped volume in z-direction 
step_size = 0.200                                   # size of the grid steps

shift_x = 0                                         # shift of the mapped volume in x-direction
shift_y = 0                                         # shift of the mapped volume in y-direction
shift_z = 0                                         # shift of the mapped volume in z-direction

# Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
target_point_coord_exp, B_target_point_exp, _ = fkt.load_data_from_folder(folder_path_points_no_current, folder_path_points_no_current, L_x, L_y, L_z, step_size, shift_x, shift_y, shift_z)

## Plot the extracted Data

Plot each $B$-field component as well as the norm of the $B$-field of both maps (with and without coil current) directly next to each other for better comparison.

In [ ]:
x = target_point_coord_exp[:,0]
y = target_point_coord_exp[:,1]
z = target_point_coord_exp[:,2]

Bnorm = np.linalg.norm(B_target_point_exp, axis=1)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scat = ax.scatter(x, y, z, c=Bnorm, s=50, cmap='viridis')

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_zlabel('Z (m)')
ax.set_title('Magnetic Field Strength at Target Points (No Current)')
fig.colorbar(scat, ax=ax, label='|B| (T)')

plt.show()

## Data definition

In this block, all experiment data which is not extracted from the ```.npz``` -file is defined.
- MSR dimansions (including dimensions of the door and inner wood-faces)
- Parameters for coil layup
- Parameters for mesh generation

(generally only SI-base units)!

In [ ]:
# Inner shield (Data from CAD model "TUM - innen.step" located in P:\MSR\CAD\STEP)
shield_height = 2.341       # height of the inner most shield (2.341)
shield_width = 2.448        # width and depth of the inner most mu-metal layer (2.448)
shield_thickness = 1e-3     # thickness of the shield walls (1e-3)

# Door (Data from CAD model "TUM - innen.step" located in P:\MSR\CAD\STEP)
door_removal = True         # Boolean which states, if the door shall be seperated in the mesh or not
door_width = 0.95           # (width of the door)
door_height = 2.004         # (height of the door)
door_offset_x = 0.2375      # (distance from door frame to right wall)
door_floor_offset = 0.0     # (distance from door frame to floor)

# Inner wood structure (Data from CAD model "TUM - innen.step" located in P:\MSR\CAD\STEP)
height_inner_wood = 2.211   # height of the inner wooden structure
width_inner_wood = 2.356    # width of the inner wooden structure
depth_inner_wood = 2.356    # depth of the inner wooden structure

# Coil Layout
coil_plane_dist_to_origin_x = width_inner_wood/2    # distance of the coil plane to the origin in x-direction
coil_plane_dist_to_origin_y = depth_inner_wood/2    # distance of the coil plane to the origin in y-direction
coil_plane_dist_to_origin_z = height_inner_wood/2   # distance of the coil plane to the origin in z-direction
useable_length_x = width_inner_wood - 0.1           # useable length, on which the coils can be placed, in x-direction
useable_length_y = depth_inner_wood - 0.1           # useable length, on which the coils can be placed, in y-direction
useable_length_z = height_inner_wood - 0.1          # useable length, on which the coils can be placed, in z-direction
n_windings = 10                                     # number of windings per coil
current = 0.001                                     # current in A flowing through the coils

# Note: One can give a scalar input for n_windings or current, which will be applied to every coil
#       or a list of integer numbers, which need to has the same length as coils present in the layout (if the shapes do not match, an error accours!)

# Note: If you only want to have one coil in one direction, set acourding distance to zero and useable_length slightly larger than diameter
# Note: If you want NO coil in one direction, do not set useable_length_{x,y,z} to zero but set {x,y,z}_dist to useable_length_{x,y,z}

# Mesh parameters (chosen due to computation performance):
num_calc_target_points_fine = 21        # number of target points of the magnetic field in each direction(25)
num_calc_target_points_coarse = 5       # number of target points of the magnetic field in each direction (5)
safety_distance = 0.05                  # Distance between coil plane and closest target point (needed to avoid errors from the assumption of infinite permeability) (0.05)
grid_quality_coilplane = 8              # number of initial nodes in one direction of the coil-plane (coarsened mesh) (12)
grid_quality_mu_metal = 8               # number of initial nodes in one direction of the mu_material plane (12)
Steps = 20                              # number of steps in the streamfunction, which define the current in the wires
refinement_factor=200                   # number of gridpoints for a refined grid in which the contourlines (wire-paths) are described

## Create Coil_Layup instance
Create instance to the class Coil_Layup to later create a mesh on this surface

In [ ]:
# Initialise coil layup class
Coils = Coil_Layup(coil_diameter = 0, x_dist = 2, y_dist = 2, z_dist = 2, n_windings = n_windings, current = current, coil_plane_dist_to_origin_x = coil_plane_dist_to_origin_x, coil_plane_dist_to_origin_y = coil_plane_dist_to_origin_y, coil_plane_dist_to_origin_z = coil_plane_dist_to_origin_z, usable_length_x = useable_length_x, usable_length_y = useable_length_y, usable_length_z = useable_length_z)

## Create mesh of coil plane

In [ ]:
Coils.create_mesh(grid_quality_coilplane, door_removal, door_width, door_height, door_floor_offset, door_offset_x)      # Create a triangular mesh on the plane where the coils are located

# # Plot the mesh in 2D
# x = Coils.mesh_back.vertices[:, 0]
# z = Coils.mesh_back.vertices[:, 2]
# faces = Coils.mesh_back.faces

# triang = mtri.Triangulation(x, z, faces)
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.triplot(triang, color='k', linewidth=0.8)
# ax.set_aspect('equal')
# ax.set_title('XZ wall with door')
# plt.show()

# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# inner_idx=bfieldtools.utils.find_mesh_boundaries(Coils.total_planes)
# inner_vertex_idx=inner_idx[0]
# for idx in inner_vertex_idx:
#     boundary=Coils.total_planes.vertices[idx]
#     ax.scatter(boundary[:,0],boundary[:,1],boundary[:,2])

# fig = plt.figure(figsize=(8,6))
# ax = fig.add_subplot(111, projection='3d')
# #fig.subplots_adjust(left=0.0, right=30.0, bottom=0.0, top=1.0)
# verts=Coils.total_planes.vertices
# ax.scatter(verts[:,0],verts[:,1],verts[:,2],alpha=0.1,label='coil planes')
# ax.scatter(target_point_coord_exp[:,0],target_point_coord_exp[:,1],target_point_coord_exp[:,2],label='MSR Map Points')
# # Axis labels
# ax.set_xlabel('x [m]')
# ax.set_ylabel('y [m]')
# ax.set_zlabel('z [m]')
# plt.legend()
# plt.tight_layout()
# plt.show()

## Shield meshing class

To account for the $\mu$-metal a meshed geometry accourding to the specified geometric data can be created by calling the class ```Mu_material``` from the file **Externe_Classes.py**. This will output a triangular mesh of the $\mu$-metal surface.

This block also plots the mesh (of one planar surface), the mesh boundaries (should be an empty plot) as well as the cells in 3D just as in the original programm.

In [ ]:
mu_grid = Mu_material(grid_quality_mu_metal, shield_height, shield_width, shield_thickness)          # Create a triangular mesh at the plane where the inner shell of mu-metal is located

# # Plot
# # plot meshboundaries of the mu-metal
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# inner_vertex_idx_shield = mu_grid.inner_idx[0]
# for idx in inner_vertex_idx_shield:
#     boundary=mu_grid.total_shield.vertices[idx]
#     ax.scatter(boundary[:,0],boundary[:,1],boundary[:,2])

# # plot vertices
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# verts_shield = mu_grid.total_shield.vertices
# ax.scatter(verts_shield[:,0],verts_shield[:,1],verts_shield[:,2],alpha=0.5)

# # plot vertices of the shield and points inside the shield
# points_inside = mu_grid.total_shield.vertices - mu_grid.thickness * mu_grid.total_shield.vertex_normals
# fig = plt.figure()
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(verts_shield[:,0],verts_shield[:,1],verts_shield[:,2],alpha=0.3)
# ax.scatter(points_inside[:,0],points_inside[:,1],points_inside[:,2],alpha=0.3)

## Finding the Coupling-matrix $C_{ij}$

**Prerequisites**:

In the calculatetion we are using the stream function $\psi(r)$ which can be closely linked to the surface current with
\begin{gather}
    {j}({r}) = \nabla_\parallel \psi(r) \times n(r)\\
    \psi(r) - \psi(r_0) = \int_{r_0}^r j(r') \cdot (\text{d}l \times n')
\end{gather}
where current_density $j$, normal-vector $n$ and position $r$ are vectors in 3D and the stream function $\psi$ is a scalar quantity.

In the programm we define the stream function as the sum of as weight $s_i$ times a hat-function $h_i$ at every vertex $i$. The hat function is a simple function which is 1 at the vertex $i$ and zero at all other verteces. Between the verteces it gets interpolated linearly!

\begin{equation}
    \psi(r) = \sum_{i=0}^{\text{number of vertices}} s_i \cdot h_i(r)
\end{equation}

Following this we will express all obperations with the stream function $\psi$ as functions of its weights $s_i$!

**Workflow of the program**:
- Calculate scalar potential matrices $U_{coil}$ and $U_{shield}$:
\begin{equation}
    \Phi(r_i) = \sum_{j=i}^{N_{vertices}} U_{ij, \text{coil}}s_{j, \text{coil}} = -\sum_{j=i}^{N_{vertices}} U_{ij, \text{shield}}s_{j, \text{shield}}
\end{equation}
where $\Phi(r_i)$ is the scalar potential, $s_j$ is the stream function at vertex $j$ and U_{ij} is the scalar potential matrix. (See paper: Magnetic field modeling with surface currents. Part II. Implementation and usage of bfieldtools; Equation (12))

- From scalar potential matrices $U_{ij}$ compute coil-shiel-coupling matrix $M_{ij}$
\begin{gather}
    U_{shield} \cdot M_{tot\_shield} = U_{coils} \\
    s_{\text{shield}} = - U_{\text{shield}}^{-1} \cdot U_{\text{coil}} \cdot s_{\text{coil}} = M_{tot\_shield} \cdot s_{\text{coil}}
\end{gather}

- Calculate the magnetic coupling matrix $C_{ij\alpha}$ of the $\mu$-metal in all three space directions
\begin{align}
    B_{\alpha,\text{shield}}(r_k) = \sum_{i=1}^{N_{vertices}} C_{ki\alpha,\text{shield}}s_{i,\text{shield}} && \alpha \in \{x,y,z\}
\end{align}
where $B_{\alpha,\text{shield}}(r_k)$ is the magnetic field at $r_k$ in direction $\alpha$ produced by the shield, and $s_{i,\text{shield}}$ is the streamfunction in the shield at vertex $i$.

- Calculate the magnetic coupling matrix $C^*_{ij\alpha,\text{shield}}$ of the $\mu$-metal as if it was caused by the coil-plane (change of reference)
 \begin{equation}
    C^*_{\alpha,\text{shield}} = C_{\alpha,\text{shield}} \cdot M_{tot\_shield}
 \end{equation}

- Calculate the magnetic coupling matrix $C^*_{ij\alpha,\text{coil}}$ of the coils
\begin{align}
    B_{\alpha,\text{coil}}(r_k) = \sum_{i=1}^{N_{vertices}} C_{ki\alpha,\text{coil}}s_{i,\text{coil}} && \alpha \in \{x,y,z\}
\end{align}
where all quantities are analog to those of the shield case.

- At the boundaries of the coil plane, it holds that in any case no current can flow over the edge, resulting in a streamfunction which is constant and can be set to zero (gauge). (**Note**, that since the shield of the MSR is connected at the edges, allowing current-flow over the edge. Therefore the edges of the shield are no boundaries! This is done in the class method

- Optain the total coupling matrix $C_{ij\alpha, \text{total}}$ of both coils and shield by adding both together
\begin{equation}
    C_{ij\alpha, \text{total}} = C^*_{ij\alpha,\text{shield}} + C_{ij\alpha,\text{coil}}
\end{equation}

- Reshape the coil coupling matrix $C^*_{ij\alpha,\text{coil}}$ and the total coupling matrix $C_{ij\alpha, \text{total}}$ as well as the predicted $B$-field due to the coils alone to better use it later on (shape: N, 3, M -> 3N, M)



In [ ]:
# Run these few lines in extra code block since they take a while to execute!
# Scalar-Potential Coupling Matrix U_{ij} (only dependent on geometrical data of meshes)
    # mu-material
U_coupling_shield_inside = bfieldtools.mesh_magnetics.scalar_potential_coupling(mu_grid.total_shield, mu_grid.points_inside)
    # Coils
U_coupling_coils_inside = bfieldtools.mesh_magnetics.scalar_potential_coupling(Coils.total_planes, mu_grid.points_inside)

# total shield coupling which links the coil plane to the shield plane U_shield * M = U_coil
Coil_Shield_coupling = np.linalg.solve(U_coupling_shield_inside, U_coupling_coils_inside)

# B-field coupling matrix C_shield of the shield alone (only dependent on gemetrical data of mu-mesh and target point coordinates)
Coupling_shield = bfieldtools.mesh_magnetics.magnetic_field_coupling(mu_grid.total_shield, target_point_coord_exp)

In [ ]:
# B-field coupling matrix C*_shield of the shield alone, seen as a secondary source from the coil plane perspective C*_shield = C_shield * M
secondary_Coupling_shield = np.tensordot(Coupling_shield, Coil_Shield_coupling, axes=([2],[0]))

# B-field coupling matrix C_coil of the coil plane (only dependent on gemetrical data of coil-mesh and target point coordinates)
Coupling_coil = bfieldtools.mesh_magnetics.magnetic_field_coupling(Coils.total_planes, r = target_point_coord_exp)

# Note: Not sure if this is correct!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! -> see Markdown above!
inner_idx = bfieldtools.utils.find_mesh_boundaries(Coils.total_planes)
inner_vertex_idx = inner_idx[0]
for idx in inner_vertex_idx:
    Coupling_coil[:, :, idx] = 0

# Combined coupling matrix C_total which incorporates the effect of the shield as a secondary source and the coil plane as a primary source
total_Coupling = Coupling_coil + secondary_Coupling_shield # Note: The minus here is directly from the equation in the paper (see markdown)

# Suppose C_coil has shape (N, 3, M) (N...number of target points, M...number of vertices in coil plane)
N, _, M = Coupling_coil.shape
Coupling_coil_flat = Coupling_coil.reshape(3 * N, M)

# Suppose C_tot has shape (N, 3, M) (N...number of target points, M...number of vertices in coil plane)
N, _, M = total_Coupling.shape
total_Coupling_flat = total_Coupling.reshape(3 * N, M)

# Suppose B_predicted has shape (N, 3) (N...number of target points)
N, _ = B_target_point_exp.shape
B_target_point_exp_flat = B_target_point_exp.reshape(3 * N)

# Debug
print(f'Shapes of the different matrices:\n   U_coupling_shield_inside: {U_coupling_shield_inside.shape}\n   U_coupling_coils_inside: {U_coupling_coils_inside.shape}\n   Coil_Shield_coupling: {Coil_Shield_coupling.shape}\n   B_coupling_shield: {Coupling_shield.shape}\n   secondary_Coupling_shield: {secondary_Coupling_shield.shape}\n   Coupling_coil: {Coupling_coil.shape}\n   total_Coupling: {total_Coupling.shape}\n   Coupling_coil_flat: {Coupling_coil_flat.shape}\n   total_Coupling_flat: {total_Coupling_flat.shape}')

## Plot streamfunction of coil

Here, we want to derive the streamfunction from:
\begin{equation}
    \mathbf{B} = \mathbf{C} \cdot \mathbf{s}
\end{equation}
Sadly, $\mathbf{C}$ is in general not a square matrix and not invertable. Therefore we can solve the problem using the numpy least square algorythm which searches for the solution $s_{LQ}$ which minimises:
\begin{equation}
    \min_{s}||Cs_{LQ} - B||_2^2
\end{equation}

However, this may lead to an overfitting of the actual streamfunction. In an effort to decrese the error which is generated by our solution, we accept fast and unphysical changes in the streamfunction. To solve this issue, we introduce a *Tikhonov* (Laplace) regularisation. This regularisation uses the *Laplace Operator* $\text{L}$, which can be seen as second derivative of our scalar field. It is added in the cost function to penalise fast changes in the gradient of the streamfunction and thereby reduces overfitting. The improved cost-function looks like
\begin{equation}
    \min_{s}||Cs_{LQ} - B||_2^2 + \lambda_{L}||\text{L} s||_2^2 + \lambda_{\Omega} s^T R s + \lambda_{\mathcal{L}} s^T \mathcal{L} s
\end{equation}
where $\lambda_{L}$, $\lambda_{\Omega}$ and $\lambda_{\mathcal{L}}$ are the constant weighing factors which prioritise smoothnmess, short wire length and low enegry stored in magnetic fields due to self_inductance. Alternatively the problem can be formulated as
\begin{equation}
    \min_s \left\lVert \begin{bmatrix} C \\ \sqrt{\lambda_{L}}L \\ \sqrt{\lambda_{\Omega}}R^{1/2} \\ \sqrt{\lambda_{\mathcal{L}}}\mathcal{L}^{1/2} \end{bmatrix} s - \begin{bmatrix} B \\ 0 \\ 0 \\ 0 \end{bmatrix}\right\rVert_2^2
\end{equation}

The problem of minimizing this function is soved by setting the first derivative to zero. Therefore the square terms need to be expanded. The derivative and subsequent equation is
\begin{equation}
    (C^TC + \lambda_{L} L^T L + \lambda_{\Omega} R + \lambda_{\mathcal{L}} \mathcal{L}) s = C^T B_{target}
\end{equation}

For the discretization of these operators see **bfieldtools documentation**!

*Note*, that for increased computational performance, sparse matrices are used! If the *self_inductance* is omitted, this leads to significant performance increse!

Also *note* that the streamsfunction needs to be zero at all boundaries!!!

In [ ]:

# 1. Mesh and MeshConductor
# cube_mesh: your trimesh.Trimesh with 6 disconnected planar faces
coil = Coils.mesh_conductor

# 2. Find boundary edges (edges that belong to only one face)
boundary_edge_indices = Coils.coil_plane_boundaries
boundary_edges = coil.mesh.edges[boundary_edge_indices]
boundary_vertices = np.unique(boundary_edges.flatten())

print(f"Total vertices: {len(coil.mesh.vertices)}")
print(f"Boundary vertices: {len(boundary_vertices)}")

# Create mask for interior vertices
is_interior = np.ones(len(coil.mesh.vertices), dtype=bool)
is_interior[boundary_vertices] = False
interior_vertices = np.where(is_interior)[0]


# 3. Regularization operators
# 3. Keep all matrices sparse (no toarray())
Coupling_coil_flat_sparse = sp.csr_matrix(Coupling_coil_flat) # made sparse
L_sparse = laplacian_matrix(coil.mesh)  # Already sparse
R_sparse = resistance_matrix(coil.mesh, sheet_resistance=1.0)  # Already sparse
L_ind_sparse = self_inductance_matrix(coil.mesh, quad_degree=3, approx_far=True)

# 4. Regularization weights (tune these)
lam_lap  = 0*2e-6                                             # Laplace penalty -> promotes mathematically smooth stream function (small curvatue of stream function)
lam_ohm  = 0*2e-6                                             # Ohmic penalty -> promotes short wire lengths 
lam_ind  = 2e-6                                             # Inductance penalty -> promotes low inductance coils (which are easier to drive and less prone to parasitic capacitances)

# 5. Solve normal equations with all three penalties
A = Coupling_coil_flat_sparse.T @ Coupling_coil_flat_sparse
A += lam_lap * (L_sparse.T @ L_sparse)
A += lam_ohm * R_sparse
A += lam_ind * L_ind_sparse

b = Coupling_coil_flat_sparse.T @ B_target_point_exp_flat

# 6. Extract submatrices for interior DOFs only
A_interior = A[np.ix_(interior_vertices, interior_vertices)]
b_interior = b[interior_vertices]

# Solve for interior stream function values
stream_func_interior = spl.spsolve(A_interior, b_interior)

# Create full stream function with zeros at boundary
stream_func_coil = np.zeros(len(coil.mesh.vertices))
stream_func_coil[interior_vertices] = stream_func_interior
# Boundary values remain 0 (enforced Dirichlet BC)

# 7. Plot the stream function on the coil mesh
fkt.plot_stream_function(Coils.total_planes.vertices, stream_func_coil)

# Debug: Print shapes and types of matrices
print(type(Coupling_coil_flat), Coupling_coil_flat.shape)
print(type(Coupling_coil_flat_sparse), Coupling_coil_flat_sparse.nnz)
print(type(L_sparse), L_sparse.shape, L_sparse.nnz)
print(type(R_sparse), R_sparse.shape, R_sparse.nnz)
print(type(L_ind_sparse), L_ind_sparse.shape)

## Obtain coil Layup

This is done by finding the isocontours of the stream function. To obtain a good reolution, the stream function is interpolated onto a more refined, square grid (to give the contour lines more points). The stream function is a measure by how much current a certain area is enclosed. Therefore we define levels which split the range of the stream function into parts (np.arange() takes a fixed step size (fixed current) and creates a certain amount of levels; np.linspace() creates a fixed amount of levels (wires) with a certain current!). The positions, where the streamfunction crosses these values are computed at the edges of each cell-surface. There the position is interpolated only along the edge. The resulting coordinates are stored.

In [ ]:
all_contours = fkt.find_all_contours(stream_func_coil, Coils.total_planes.vertices, Steps, refinement_factor)

print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
print(f"Total planes: {len(all_contours)}")
for face_idx in range(len(all_contours)):
    n_contours = len(all_contours[face_idx])
    print(f"Plane {face_idx + 1}: {n_contours} contours")
    for contour_idx in range(n_contours):
        n_points = all_contours[face_idx][contour_idx].shape[0]
        print(f"  Contour {contour_idx + 1}: {n_points} points")

print(f"\nData structure:")
print(f"  all_contours shape: ({len(all_contours)}, num_contours, num_points_in_contour, 3)")

In [ ]:
fkt.plot_contours(all_contours, stream_func_coil, Coils.total_planes.vertices, Steps)

# Evaluation

Make nice plots and Sanity checks here!

## total streamfunction on coil plane

Compute the equivalent total streamfunction on the coil_plane. This is done to make sanity chacks later on but is also very memory intensive, since the streamfunction is calculated via the inverse of the problem:

\begin{equation}
    B_{i, \alpha} = C_{i, j, \alpha} \cdot \psi_j
\end{equation}

In [ ]:
if True: # Only run if of interest (This block takes a while)
    # For the sake of completeness, the equivalent total stream function is computed on the coil-planes
    stream_func_tot, _, _, _ = np.linalg.lstsq(total_Coupling_flat, B_target_point_exp_flat, rcond=None)

    # Debug
    print(f'The shape of the total streamfunction is: {stream_func_tot.shape}')

    fkt.plot_stream_function(Coils.total_planes.vertices, stream_func_tot)

## $B$-field by streamfunction

This $B$-field should be close to the one we actually measured, since it is derived from the stream function we used.

In [ ]:
# Use coil stream function and total coupling matrix to compute total magnetic field
B_check_flat = total_Coupling_flat @ stream_func_tot

# Extract values in B_check_flat of shape (3*number_target_points) into B_check of shape (number_target_points, 3)
N, _ = B_target_point_exp.shape       # N = number_of_target_points
B_check = B_check_flat.reshape(N, 3)    # B_check has shape (number_target_points, 3)

# Debug
print(f'Shapes of the different matrices:\n   Stream function of coils: {stream_func_coil.shape}\n   B_check_flat: {B_check_flat.shape}\n   B_check: {B_check.shape}')

# Plots
fig_B_check = plt.figure()
ax_B_check = fig_B_check.add_subplot(111, projection='3d')
sc_B_check = ax_B_check.scatter(
    target_point_coord_exp[:, 0],
    target_point_coord_exp[:, 1],
    target_point_coord_exp[:, 2],
    c=np.linalg.norm(B_check, axis=1),
    s=100,
    cmap='viridis',
    alpha = 0.8,
)
fig_B_check.colorbar(sc_B_check, ax=ax_B_check, label=r'$|\mathbf{B}_{\text{check}}|$')

## Check

Comparison between original $B$-field from experiment and $B$-field we get from derived stream function

## Strom

Berechne den Strom aus der Streamfunction um nachzuprüfen, ob der Output sinnvoll ist.

Außerdem: Darüber nachdenken, ob ich das Problem rückwerts angehen kann, um direkt ein Ergebnis zu finden, und nicht über das B-Feld zu müssen.

Allgemein gilt:
\begin{gather}
    j(r) = \nabla_\parallel \psi(r) \times n(r) \\
    \psi(r)-\psi(r_0) = \int_{r_0}^r j(r')\cdot (dl' \times n')
\end{gather}

In [ ]:
if False:
    n = int(np.sqrt(len(Coils.total_planes.vertices) / 6))
    stream_func_reshaped = np.array(stream_func_coil_coarse).reshape(6, n**2)

    x = Coils.total_planes.vertices[:, 0].reshape(6, n**2)
    y = Coils.total_planes.vertices[:, 1].reshape(6, n**2)
    z = Coils.total_planes.vertices[:, 2].reshape(6, n**2)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=False, dpi=200)
    axes = axes.ravel()

    Faces = ['Top', 'Bottom', 'Back', 'Front', 'right', 'left']

    for face_idx, ax in enumerate(axes):
        x_face = x[face_idx]
        y_face = y[face_idx]
        z_face = z[face_idx]
        s_face = stream_func_reshaped[face_idx]

        xr = np.ptp(x_face)
        yr = np.ptp(y_face)
        zr = np.ptp(z_face)

        ranges = {'x': xr, 'y': yr, 'z': zr}
        const_axis = min(ranges, key=ranges.get)

        if const_axis == 'x':
            a, b = y_face, z_face
            a_name, b_name = 'y', 'z'
        elif const_axis == 'y':
            a, b = x_face, z_face
            a_name, b_name = 'x', 'z'
        else:
            a, b = x_face, y_face
            a_name, b_name = 'x', 'y'

        sort_indices = np.lexsort((b, a))

        a_sorted = a[sort_indices].reshape(n, n)
        b_sorted = b[sort_indices].reshape(n, n)
        s_sorted = s_face[sort_indices].reshape(n, n)

        da = np.gradient(s_sorted, axis=1)
        db = -np.gradient(s_sorted, axis=0)

        sc = ax.scatter(
            a_sorted.ravel(),
            b_sorted.ravel(),
            c=s_sorted.ravel(),
            cmap='RdBu_r',
            s=50,
            alpha=0.8,
            edgecolors='none'
        )

        ax.quiver(
            a_sorted, b_sorted,
            da, db,
            alpha=0.9,
            scale=0.05
        )

        ax.set_xlabel(a_name)
        ax.set_ylabel(b_name)
        ax.set_title(Faces[face_idx])
        ax.axis('equal')

    divider = make_axes_locatable(fig.axes[-1])
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(sc, cax=cax, label='Streamfunction')
    plt.tight_layout(rect=[0, 0, 0.9, 1])
    plt.show()